In [0]:
%sql
    
create catalog if not exists my_ecommerce_store
MANAGED LOCATION '';

In [0]:
if not any(mount.mountPoint == '/mnt/my_store' for mount in dbutils.fs.mounts()):
    dbutils.fs.mount(
        source = 'wasbs://<container>@<storage_account>.blob.core.windows.net',
        mount_point = '/mnt/my_store',
        extra_configs = {
            'fs.azure.account.key.mystoarageadls001.blob.core.windows.net': ''
        }
    )
else:
    print('Mount already exists at /mnt/my_store')

Mount already exists at /mnt/my_store


In [0]:
%sql
create schema if not exists my_ecommerce_store.bronze
COMMENT 'bronze layer'
;
    
create schema if not exists my_ecommerce_store.silver
COMMENT 'silver layer';   

create schema if not exists my_ecommerce_store.gold
COMMENT 'gold layer';

In [0]:
#dbutils.fs.ls('/mnt/my_store/Ecommerce_store/')

[FileInfo(path='dbfs:/mnt/my_store/Ecommerce_store/olist_customers_dataset.csv', name='olist_customers_dataset.csv', size=9033957, modificationTime=1775892199000),
 FileInfo(path='dbfs:/mnt/my_store/Ecommerce_store/olist_order_items_dataset.csv', name='olist_order_items_dataset.csv', size=15438671, modificationTime=1775892225000),
 FileInfo(path='dbfs:/mnt/my_store/Ecommerce_store/olist_order_payments_dataset.csv', name='olist_order_payments_dataset.csv', size=5777138, modificationTime=1775892203000),
 FileInfo(path='dbfs:/mnt/my_store/Ecommerce_store/olist_orders_dataset.csv', name='olist_orders_dataset.csv', size=17654914, modificationTime=1775892223000),
 FileInfo(path='dbfs:/mnt/my_store/Ecommerce_store/olist_products_dataset.csv', name='olist_products_dataset.csv', size=2379446, modificationTime=1775892174000)]

In [0]:
df_customers = spark.read.csv('/mnt/my_store/Ecommerce_store/olist_customers_dataset.csv', header=True, inferSchema=True)
df_customers.write.mode('overwrite').saveAsTable('my_ecommerce_store.bronze.olist_customers')

df_orders = spark.read.csv('/mnt/my_store/Ecommerce_store/olist_orders_dataset.csv', header = True, inferSchema=
                            True)
df_orders.write.mode('overwrite').saveAsTable('my_ecommerce_store.bronze.olist_orders'
                                              )
df_products = spark.read.csv('/mnt/my_store/Ecommerce_store/olist_products_dataset.csv', header=True, inferSchema=True)
df_products.write.mode('overwrite').saveAsTable('my_ecommerce_store.bronze.olist_products'
                                              )
df_payments  = spark.read.csv('/mnt/my_store/Ecommerce_store/olist_order_payments_dataset.csv', header=True, inferSchema=True)
df_payments.write.mode('overwrite').saveAsTable('my_ecommerce_store.bronze.olist_order_payments')
df_order_items = spark.read.csv('/mnt/my_store/Ecommerce_store/olist_order_items_dataset.csv', header=True, inferSchema=True)
df_order_items.write.mode('overwrite').saveAsTable('my_ecommerce_store.bronze.olist_order_items')

In [0]:
# df_customers.show(10)
# df_orders.show(10)
# df_products.show(10)
# df_payments.show(10)

+--------------------+--------------------+------------------------+--------------------+--------------+
|         customer_id|  customer_unique_id|customer_zip_code_prefix|       customer_city|customer_state|
+--------------------+--------------------+------------------------+--------------------+--------------+
|06b8999e2fba1a1fb...|861eff4711a542e4b...|                   14409|              franca|            SP|
|18955e83d337fd6b2...|290c77bc529b7ac93...|                    9790|sao bernardo do c...|            SP|
|4e7b3e00288586ebd...|060e732b5b29e8181...|                    1151|           sao paulo|            SP|
|b2b6027bc5c5109e5...|259dac757896d24d7...|                    8775|     mogi das cruzes|            SP|
|4f2d8ab171c80ec83...|345ecd01c38d18a90...|                   13056|            campinas|            SP|
|879864dab9bc30475...|4c93744516667ad3b...|                   89254|      jaragua do sul|            SC|
|fd826e7cf63160e53...|addec96d2e059c80c...|            